## Acidentes de Trânsito nas Rodovias Federais Brasileiras
**Nome:** Leonardo Quederoli Leme - **RA:** 241020409<br>
**Nome:** Luís Fernando das Chagas - **RA:** 241020034

## Contexto do problema
A malha viária federal estende-se por dezenas de milhares de quilômetros, abrangendo realidades geográficas, climáticas e de tráfego profundamente heterogêneas. O enfrentamento da violência no trânsito por parte dos órgãos fiscalizadores enfrenta um gargalo estrutural: a assimetria entre a extensão territorial a ser coberta e a limitação dos recursos operacionais disponíveis, que incluem contingente de agentes, viaturas, radares móveis e etilômetros.

A distribuição tradicional da fiscalização por vezes baseou-se em percepções empíricas ou no patrulhamento ostensivo genérico, modelos que demonstram limitação para conter sinistros em janelas específicas de vulnerabilidade. A consolidação do repositório DATATRAN/PRF possibilita uma transição metodológica para a Segurança Viária Baseada em Evidências (Data-Driven Policing). Por meio da exploração sistemática desses dados e de técnicas de segmentação, torna-se viável transformar dados históricos em inteligência tática, fundamentando a tomada de decisão sobre onde e quando intervir.       


## Problema central
O problema central investigado nesta análise é a distribuição não uniforme da severidade dos acidentes ao longo da malha rodoviária. As ocorrências que resultam em lesões graves ou fatalidades não ocorrem ao acaso; elas tendem a se adensar na interseção de fatores contextuais específicos. Elementos como a transição de iluminação natural (crepúsculo e madrugada), a configuração da via (pista simples e traçados em curva), a ocorrência de intempéries (chuva e neblina) e condutas humanas de risco atuam de forma sinérgica, multiplicando a energia dos impactos.

O propósito deste relatório analítico é decompor os sinistros em suas variáveis explicativas, fornecendo ao órgão de trânsito matrizes claras de priorização para otimizar escalas de serviço, posicionamento de equipamentos e operações ostensivas focadas na mitigação de mortes.

## Limpeza e Preparação dos Dados

In [1]:
# Importação das bibliotecas necessárias
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Configura como os dataframes serão exibidos
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Carrega os dois datasets que serão utilizados
df1 = pd.read_csv('datatran2025.csv', sep=';', encoding='ISO-8859-1')
df2 = pd.read_csv('acidentes2025.csv', sep=';', encoding='ISO-8859-1')

### Preencher Dados Faltantes e Corrigir Inconsistências

In [4]:
# Somatório do total de rows NaN em cada coluna
df1.isna().sum()[df1.isna().sum() > 0]

classificacao_acidente     1
regional                   2
delegacia                 22
uop                       38
dtype: int64

In [5]:
# Somatório do total de rows NaN em cada coluna
df2.isna().sum()[df2.isna().sum() > 0]

classificacao_acidente        7
tipo_veiculo               6341
tipo_envolvido            17150
estado_fisico             17150
sexo                      17150
regional                      6
delegacia                    74
uop                         130
dtype: int64

In [6]:
# Preenche as rows com conteúdo NaN das colunas específicadas, com o termo "Não Informado"
df1[["regional", "delegacia", "uop"]] = df1[["regional", "delegacia", "uop"]].fillna("Não Informado")
df2[["regional", "delegacia", "uop", "tipo_veiculo", "tipo_envolvido", "estado_fisico", "sexo"]] = df2[["regional", "delegacia", "uop", "tipo_veiculo", "tipo_envolvido", "estado_fisico", "sexo"]].fillna("Não Informado")

In [7]:
# Determina um conjunto de condições
conditions = [
    (df1["mortos"] > 0),
    (df1["feridos"] > 0),
]

# Determina um conjunto de escolhas
choices = ["Com Vítimas Fatais", "Com Vítimas Feridas"]

# Preenche a coluna com as escolhas definidas de acordo com cada condição
# Caso nenhuma condição seja cumprida, a coluna é preenchida com o default
df1["classificacao_acidente"] = np.select(conditions, choices, default="Sem Vítimas") 

In [8]:
# Cria um mapa atribuindo a classificação do acidente ao ID da sua respectiva ocorrência
mapping = df1.set_index("id")["classificacao_acidente"]

# Preenche a coluna com os dados do mapa onde o ID da ocorrência é equivalente
df2["classificacao_acidente"] = df2["id"].map(mapping)

In [9]:
# Determina a quantidade de feridos como a soma de feridos leves e graves
df1["feridos"] = df1["feridos_leves"] + df1["feridos_graves"]

# Determina o total de pessoas como a soma de óbitos, feridos, ilesos e não informados
df1["pessoas"] = df1["mortos"] + df1["ilesos"] + df1["ignorados"] + df1["feridos"]

In [10]:
# Altera os nomes das colunas em questão
df1.rename(columns={"ignorados" : "nao_informados", "uso_solo" : "local_acidente", "condicao_metereologica" : "condicao_meteorologica"}, inplace = True)
df2.rename(columns={"uso_solo" : "local_acidente", "condicao_metereologica" : "condicao_meteorologica"}, inplace = True)

### Padronizar e Preparar Dados para Análise

In [11]:
# Define as colunas que terão seus campos padronizados
columns_to_lower_df1 = ["municipio", "causa_acidente", "tipo_acidente", "classificacao_acidente", "fase_dia", "sentido_via", "condicao_meteorologica", "tipo_pista", "tracado_via", "local_acidente"]
columns_to_lower_df2 = ["municipio", "causa_acidente", "tipo_acidente", "classificacao_acidente", "fase_dia", "sentido_via", "condicao_meteorologica", "tipo_pista", "tracado_via", "local_acidente", "tipo_veiculo", "tipo_envolvido", "estado_fisico", "sexo"]

# Torna minúsculo todos os caracteres das strings nas colunas especificadas 
df1[columns_to_lower_df1] = df1[columns_to_lower_df1].map(lambda x : x.lower())
df2[columns_to_lower_df2] = df2[columns_to_lower_df2].map(lambda x : x.lower())

In [12]:
# Formata a coluna "horario", convertendo string em time
df1["horario"] = pd.to_datetime(df1["horario"], format = "%H:%M:%S").dt.time
df2["horario"] = pd.to_datetime(df2["horario"], format = "%H:%M:%S").dt.time

In [13]:
# Remove as colunas em questão
df1.drop(columns = ["km", "regional", "delegacia", "uop"], inplace = True)
df2.drop(columns = ["pesid", "km", "id_veiculo", "marca", "sexo", "ilesos", "feridos_leves", "feridos_graves", "mortos", "regional", "delegacia", "uop"], inplace = True)

In [14]:
# Altera os campos da coluna "local_acidente" seguindo as regras de tradução definidas a seguir
local = {"sim" : "urbano", "não" : "rural"}

df1["local_acidente"] = df1["local_acidente"].map(local)
df2["local_acidente"] = df2["local_acidente"].map(local)